# DTDLINK Stage 2 — Reproducible cleaning

## Goal

Notebook này giải thích và chạy pipeline cleaning đã được kiểm thử. Kết quả chính: từ 651.191 hàng raw còn 641.113 URL duy nhất; 7 URL được trim, 12 hàng mâu thuẫn được cách ly và 10.066 bản sao bị loại.

## Setup

Thêm thư mục `src` vào Python path để notebook dùng đúng code tái sử dụng của dự án.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from dtdlink_ml.data_cleaning import run_cleaning_pipeline

## Steps

Pipeline kiểm tra raw → trim hai đầu URL → cách ly conflict → loại duplicate → thêm target → ghi output và audit.

In [2]:
cleaned, audit = run_cleaning_pipeline()

row_changes = pd.DataFrame([
    {'step': 'Raw input', 'rows': audit['input']['rows'], 'rows_removed': 0},
    {'step': 'After conflict quarantine', 'rows': audit['rules']['label_conflicts']['rows_after'], 'rows_removed': audit['rules']['label_conflicts']['rows_removed']},
    {'step': 'After exact deduplication', 'rows': audit['output']['rows'], 'rows_removed': audit['rules']['exact_duplicates']['rows_removed']},
])
row_changes

,step,rows,rows_removed
0,Raw input,651191,0
1,After conflict quarantine,651179,12
2,After exact deduplication,641113,10066


In [3]:
class_comparison = pd.DataFrame({
    'before': audit['input']['type_counts'],
    'after': audit['output']['type_counts'],
}).fillna(0).astype(int)
class_comparison['change'] = class_comparison['after'] - class_comparison['before']
class_comparison

,before,after,change
benign,428103,428074,-29
defacement,96457,95308,-1149
malware,32520,23645,-8875
phishing,94111,94086,-25


Malware giảm nhiều nhất vì phần lớn bản sao dư của Stage 1 nằm trong lớp này. Không diễn giải tỷ lệ sau cleaning như tỷ lệ URL nguy hiểm ngoài Internet.

In [4]:
conflicts = pd.read_csv(project_root / 'reports/cleaning/conflicting_records.csv')
conflicts

,source_record_number,url,type,target
0,181227,memory.loc.gov/ammem/ccmphtml/colahome.html,benign,0
1,208440,routesinternational.com/air.htm,benign,0
2,213052,en.wikipedia.org/wiki/E-book,benign,0
3,259610,en.wikipedia.org/wiki/Desktop_publishing,benign,0
4,275789,groups.yahoo.com/group/Band-in-a-Box/,benign,0
5,328666,ebookstore.sony.com/reader/,benign,0
6,609890,memory.loc.gov/ammem/ccmphtml/colahome.html,phishing,1
7,612942,routesinternational.com/air.htm,phishing,1
8,616833,en.wikipedia.org/wiki/Desktop_publishing,phishing,1
9,616867,en.wikipedia.org/wiki/E-book,phishing,1


## Checks

Các assert dưới đây kiểm tra completion gate quan trọng mà không in toàn bộ dataset.

In [5]:
assert audit['input']['sha256_before'] == audit['input']['sha256_after']
assert audit['rules']['edge_whitespace']['urls_changed'] == 7
assert audit['rules']['label_conflicts']['rows_removed'] == 12
assert audit['rules']['exact_duplicates']['rows_removed'] == 10066
assert len(cleaned) == cleaned['url'].nunique() == 641113
assert not cleaned[['url', 'type']].duplicated().any()
'All Stage 2 notebook checks passed.'

'All Stage 2 notebook checks passed.'

## Next Steps

Stage 2 chỉ chuẩn bị dữ liệu. Stage 3 sẽ xây URL parser và feature extractor, bao gồm test cho Unicode, ký tự điều khiển, URL dài và input khó parse.